<a href="https://colab.research.google.com/github/rodrigologin0-cpu/Rodrigo-de-Souza-Lima/blob/main/NARX_Otimizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NARX POLINOMIAL COM OTIMIZAÇÃO AUTOMÁTICA DE ESTRUTURA
# Testa lags, ordem polinomial e regularização
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# ============================================================
# 1. Upload do arquivo
# ============================================================

print("Faça upload do arquivo Excel (.xlsx)")
uploaded = files.upload()

arquivo = list(uploaded.keys())[0]
df = pd.read_excel(arquivo)

print("\nColunas encontradas:")
for col in df.columns:
    print("-", col)

In [ ]:
# ============================================================
# 2. Configurações do usuário
# ============================================================

col_y = input("\nDigite o nome da variável de saída Y: ")

n_inputs = int(input("Quantas entradas exógenas o modelo terá? "))

cols_u = []
for i in range(n_inputs):
    col = input(f"Digite o nome da entrada U{i+1}: ")
    cols_u.append(col)

horizonte = int(input("Quantos passos à frente deseja prever? Exemplo: 1, 5, 10: "))

max_lag_y = int(input("Máximo de atrasos da saída Y a testar. Exemplo: 5: "))
max_lag_u = int(input("Máximo de atrasos das entradas U a testar. Exemplo: 5: "))

graus = input("Graus polinomiais a testar. Exemplo: 1,2: ")
graus = [int(x.strip()) for x in graus.split(",")]

usar_lasso = input("Incluir Lasso na busca? (s/n): ").lower() == "s"

criterio = input("Critério de escolha: mae ou rmse? ").lower()
if criterio not in ["mae", "rmse"]:
    criterio = "mae"

# Penalização opcional por complexidade
penalizar_complexidade = input("Penalizar modelos muito grandes? (s/n): ").lower() == "s"

In [ ]:
# ============================================================
# 3. Função para criar base NARX
# ============================================================

def criar_base_narx(df, col_y, cols_u, n_lags_y, n_lags_u, horizonte):
    data = df.copy()

    for lag in range(1, n_lags_y + 1):
        data[f"{col_y}_t-{lag}"] = data[col_y].shift(lag)

    for col in cols_u:
        for lag in range(1, n_lags_u + 1):
            data[f"{col}_t-{lag}"] = data[col].shift(lag)

    data[f"{col_y}_target_t+{horizonte}"] = data[col_y].shift(-horizonte)

    data = data.dropna().reset_index(drop=True)

    feature_cols = [c for c in data.columns if "_t-" in c]

    X = data[feature_cols]
    y = data[f"{col_y}_target_t+{horizonte}"]

    return X, y, feature_cols, data

In [ ]:
# ============================================================
# 4. Função para treinar e avaliar
# ============================================================

def treinar_avaliar(X, y, grau, tipo_modelo, alpha=None):
    train_size = int(len(X) * 0.8)

    X_train_base = X.iloc[:train_size]
    X_test_base  = X.iloc[train_size:]

    y_train = y.iloc[:train_size]
    y_test  = y.iloc[train_size:]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_base)
    X_test_scaled  = scaler.transform(X_test_base)

    poly = PolynomialFeatures(
        degree=grau,
        include_bias=False
    )

    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly  = poly.transform(X_test_scaled)

    if tipo_modelo == "Linear":
        modelo = LinearRegression()

    elif tipo_modelo == "Ridge":
        modelo = Ridge(alpha=alpha)

    elif tipo_modelo == "Lasso":
        modelo = Lasso(alpha=alpha, max_iter=20000)

    modelo.fit(X_train_poly, y_train)

    y_pred_train = modelo.predict(X_train_poly)
    y_pred_test  = modelo.predict(X_test_poly)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)

    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_test = r2_score(y_test, y_pred_test)

    nomes_features = poly.get_feature_names_out(X.columns)

    return {
        "modelo_obj": modelo,
        "scaler": scaler,
        "poly": poly,
        "nomes_features": nomes_features,
        "y_test": y_test,
        "y_pred_test": y_pred_test,
        "mae_train": mae_train,
        "rmse_train": rmse_train,
        "r2_train": r2_train,
        "mae_test": mae_test,
        "rmse_test": rmse_test,
        "r2_test": r2_test,
        "n_termos": len(nomes_features)
    }


In [ ]:
# ============================================================
# 5. Busca automática
# ============================================================

resultados = []
melhor = None

alphas_ridge = [0.01, 0.1, 1, 10, 100]
alphas_lasso = [0.0001, 0.001, 0.01, 0.1, 1]

for n_lags_y in range(1, max_lag_y + 1):
    for n_lags_u in range(1, max_lag_u + 1):
        for grau in graus:

            X, y, feature_cols, data_modelo = criar_base_narx(
                df, col_y, cols_u, n_lags_y, n_lags_u, horizonte
            )

            modelos_teste = [("Linear", None)]

            for a in alphas_ridge:
                modelos_teste.append(("Ridge", a))

            if usar_lasso:
                for a in alphas_lasso:
                    modelos_teste.append(("Lasso", a))

            for tipo_modelo, alpha in modelos_teste:

                try:
                    r = treinar_avaliar(X, y, grau, tipo_modelo, alpha)

                    score_base = r["mae_test"] if criterio == "mae" else r["rmse_test"]

                    if penalizar_complexidade:
                        score = score_base * (1 + 0.001 * r["n_termos"])
                    else:
                        score = score_base

                    linha = {
                        "tipo_modelo": tipo_modelo,
                        "alpha": alpha,
                        "lags_y": n_lags_y,
                        "lags_u": n_lags_u,
                        "grau": grau,
                        "n_termos": r["n_termos"],
                        "mae_train": r["mae_train"],
                        "rmse_train": r["rmse_train"],
                        "r2_train": r["r2_train"],
                        "mae_test": r["mae_test"],
                        "rmse_test": r["rmse_test"],
                        "r2_test": r["r2_test"],
                        "score": score
                    }

                    resultados.append(linha)

                    if melhor is None or score < melhor["score"]:
                        melhor = linha.copy()
                        melhor["objetos"] = r
                        melhor["X"] = X
                        melhor["y"] = y
                        melhor["feature_cols"] = feature_cols

                except Exception as e:
                    print("Erro em combinação:", n_lags_y, n_lags_u, grau, tipo_modelo, alpha, e)

In [ ]:
# ============================================================
# 6. Resultados da busca
# ============================================================

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values("score").reset_index(drop=True)

print("\nTop 10 modelos encontrados:")
display(df_resultados.head(10))

print("\nMelhor modelo:")
print(f"Modelo: {melhor['tipo_modelo']}")
print(f"Alpha: {melhor['alpha']}")
print(f"Lags Y: {melhor['lags_y']}")
print(f"Lags U: {melhor['lags_u']}")
print(f"Grau polinomial: {melhor['grau']}")
print(f"Nº termos: {melhor['n_termos']}")
print(f"MAE teste: {melhor['mae_test']:.5f}")
print(f"RMSE teste: {melhor['rmse_test']:.5f}")
print(f"R² teste: {melhor['r2_test']:.5f}")



In [ ]:
# ============================================================
# 7. Gráficos do melhor modelo
# ============================================================

n_plot = int(input("\nQuantas amostras deseja mostrar no gráfico? Exemplo: 500: "))

y_test = melhor["objetos"]["y_test"].values
y_pred = melhor["objetos"]["y_pred_test"]

plt.figure(figsize=(14, 6))
plt.plot(y_test[:n_plot], label=f"Real {col_y}(t+{horizonte})")
plt.plot(y_pred[:n_plot], linestyle="--", label=f"Previsto {horizonte} passos à frente")
plt.title(
    f"Melhor NARX Polinomial | {melhor['tipo_modelo']} | "
    f"LagsY={melhor['lags_y']} LagsU={melhor['lags_u']} Grau={melhor['grau']}"
)
plt.xlabel("Amostras do teste")
plt.ylabel(col_y)
plt.legend()
plt.grid(True)
plt.show()

erro = y_test - y_pred

plt.figure(figsize=(14, 4))
plt.plot(erro[:n_plot])
plt.axhline(0, linestyle="--")
plt.title(f"Erro de previsão - {horizonte} passos à frente")
plt.xlabel("Amostras do teste")
plt.ylabel("Erro")
plt.grid(True)
plt.show()



In [ ]:
# ============================================================
# 8. Exportar coeficientes do melhor modelo
# ============================================================

modelo = melhor["objetos"]["modelo_obj"]
nomes_features = melhor["objetos"]["nomes_features"]

coeficientes = pd.DataFrame({
    "Termo": nomes_features,
    "Coeficiente": modelo.coef_
})

coeficientes["Abs"] = coeficientes["Coeficiente"].abs()
coeficientes = coeficientes.sort_values("Abs", ascending=False)

print("\nBias / Intercepto:")
print(modelo.intercept_)

print("\nPrincipais coeficientes:")
display(coeficientes.head(30))

coef_export = pd.DataFrame({
    "Termo": nomes_features,
    "Coeficiente": modelo.coef_
})

coef_export.loc[len(coef_export)] = ["Bias", modelo.intercept_]

df_resultados.to_excel("resultado_busca_narx.xlsx", index=False)
coef_export.to_excel("coeficientes_melhor_narx.xlsx", index=False)

files.download("resultado_busca_narx.xlsx")
files.download("coeficientes_melhor_narx.xlsx")

print("\nArquivos gerados:")
print("- resultado_busca_narx.xlsx")
print("- coeficientes_melhor_narx.xlsx")